# שיעור 10 - סוכני בינה מלאכותית בפרודקשן

בשיעור זה תלמדו **דפוסי פרודקשן** לסוכני בינה מלאכותית באמצעות Microsoft Agent Framework (Python). נסקור:

- **תצפית** — הוספת תזמון ורישום לאינטראקציות עם הסוכן
- **הערכה** — שימוש בסוכן מעריך כדי לדרג את איכות התגובה
- **ניהול עלויות** — אסטרטגיות לאופטימיזציית טוקנים ובחירת מודלים

התסריט הוא **סוכן נסיעות** שעוזר למשתמשים לתכנן טיולים, עם ניטור והערכה כחלק מעל.


## הגדרה


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## שיקולים לפרודקשן

להעביר סוכני AI מפרוטוטיפים לפרודקשן דורש תשומת לב קפדנית לשלושה עמודים:

1. **יכולת ניתוח ומעקב** — יש צורך בנראות לגבי מה שהסוכן עושה, כמה זמן זה לוקח, ואילו כלים הוא קורא. בלי מעקב ורישום, איתור בעיות בפרודקשן הוא כמעט בלתי אפשרי.

2. **הערכה** — בדיקות איכות אוטומטיות מבטיחות שהתגובות של הסוכן נשארות מדויקות, שלמות ומועילות לאורך זמן. סוכן מעריך יכול לדרג תגובות לפי קריטריונים מוגדרים.

3. **ניהול עלויות** — שימוש בטוקנים משפיע ישירות על העלות. אסטרטגיות כמו אופטימיזציה של הפרומפט, בחירת מודל, והאחסון במטמון עוזרות לשמור על עלויות תחת שליטה מבלי לוותר על איכות.


## בניית סוכן נצפה

אנחנו מגדירים כלי טיולים ועוטפים את קריאת הסוכן עם מדידת זמן כדי שנוכל לעקוב אחר ההשהיה. בסביבת הייצור הייתם משתלבים עם OpenTelemetry או מערך מעקב דומה.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## דפוסי הערכה

דפוס נפוץ בייצור הוא להשתמש בסוכן שני כ**מוערך**. המוערך מדרג את תגובת הסוכן הראשי על פי קריטריונים שהוגדרו מראש כמו שלמות, דיוק, ועזרה.

זה מאפשר:
- שערי איכות אוטומטיים לפני שהתשובות מגיעות למשתמשים
- גילוי רגרסיות כאשר ההנחיות או הדגמים משתנים
- ניטור מתמשך של ביצועי הסוכן לאורך זמן


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## אסטרטגיות לניהול עלויות

שליטה בעלויות היא קריטית עבור סוכני AI למוצר. הנה אסטרטגיות מפתח:

| אסטרטגיה | תיאור |
|---|---|
| **אופטימיזציה של הפרומפט** | שמור על הוראות המערכת תמציתיות. הסר הקשר מיותר כדי להפחית את כמות הטוקנים בקלט. |
| **בחירת מודל** | השתמש במודלים קטנים וזולים יותר (למשל GPT-4o-mini) למשימות פשוטות כמו סיווג או חילוץ, ושמור מודלים גדולים יותר להיגיון מורכב. |
| **מטמון** | שמור בתור מטמון תוצאות כלים ושאילתות תכופות כדי להימנע מקריאות API מיותרות. |
| **תקציבי טוקנים** | הגדר הגבלות `max_tokens` כדי למנוע תגובות ארוכות בלתי צפויות. |
| **ביצוע באצוות** | קבץ שאילתות משתמש מרובות לקריאת API אחת כאשר ניתן. |

בפועל, גישה בשכבות עובדת היטב: נהל בקשות פשוטות למודל מהיר וזול ועלה רק עם שאילתות מורכבות למודל בעל יכולת גבוהה יותר.


## סיכום

במהלך השיעור למדת כיצד:

1. **להוסיף יכולת תצפית** לאינטראקציות עם הסוכן באמצעות תזמון ותיעוד, ולבסס את התשתית למעקב וניטור.
2. **להעריך תגובות סוכן** באופן אוטומטי בעזרת סוכן מעריך שמדרג שלמות, דיוק ומועילות.
3. **לנהל עלויות** באמצעות אופטימיזציית הפקודות, בחירת מודלים, מטמון ותקציבי טוקנים.

תבניות ייצור אלה עוזרות להבטיח שהסוכנים שלך יהיו אמינים, מדידים וכלכליים בקנה מידה.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
